# Chapter 2 — VAR Models: Empirical Examples
## VAR Estimation · Cointegration & VECM · The Identification Problem

**Macroeconometrics** | Alessia Paccagnini | Companion Notebook

---

This notebook replicates all empirical examples from Chapter 2 using real quarterly US macroeconomic data from the **FRED-QD dataset** (McCracken & Ng). All three examples share a single data loading step and then branch into independent analyses.

| Example | Topic | Core output |
|---|---|---|
| **1** | VAR Estimation, Lag Selection, Granger Causality | Information criteria plot · Residual diagnostics · Granger table |
| **2** | Cointegration & VECM (Consumption-Income) | Johansen trace/max-eigenvalue tests · Long-run OLS · Equilibrium error |
| **3** | The Identification Problem | Reduced-form innovation correlations · Unaided IRFs + ⚠ interpretation warning |

**Design conventions:**
- 🎛️ cells mark tunable parameters — change and re-run freely
- Every code block has a **header comment** explaining *why* this step is done
- **Inline comments** on every non-trivial line explain *what* it does
- Every figure is followed by a **"What to look for"** note
- Figures saved as `.png` and `.pdf` for LaTeX/Overleaf

**Data source:** FRED-QD, `2026-01-QD.xlsx`, quarterly, 1960:Q2 – 2026:Q1

---

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy import stats
import warnings, os
warnings.filterwarnings('ignore')

# statsmodels — VAR, VECM, Johansen, Granger, diagnostics
from statsmodels.tsa.api import VAR, VECM
from statsmodels.tsa.vector_ar.vecm import coint_johansen
from statsmodels.tsa.stattools import adfuller, grangercausalitytests
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.stats.stattools import durbin_watson
from statsmodels.graphics.tsaplots import plot_acf

# ── Output directory ──────────────────────────────────────────────────────────
OUT_DIR = '.'
os.makedirs(OUT_DIR, exist_ok=True)

# ── Plot style ────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 130,
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 11,
    'axes.titleweight': 'bold',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'legend.fontsize': 10,
})

print('All packages loaded.')

---
# Data Loading
*Shared by all three examples*

## About the FRED-QD dataset

The **FRED-QD** dataset (McCracken & Ng, 2016, 2020) provides a balanced panel of ~250 quarterly US macroeconomic and financial series from FRED, updated regularly. It is the standard data source for large-scale empirical macro work.

**File structure of `2026-01-QD.xlsx` (sheet `in`):**

| Row | Content |
|---|---|
| 0 | Column names (variable codes, e.g. `GDPC1`, `CPIAUCSL`) |
| 1 | Factor grouping code |
| 2 | Suggested transformation code (1=levels, 5=log-diff×100, etc.) |
| 3 | 1959:Q1 (first data row) |
| 4 | 1959:Q2 |
| 5 | 1959:Q3 ← we start here (`iloc[5:]`) |
| … | … |

We need 1959:Q3 onward because computing quarterly growth rates requires one lag, so the *transformed* series starts at 1959:Q4. After a further lag-drop inside the VAR, the estimation sample begins at **1960:Q2**.

**Variables used:**

| Code | Description | Transformation |
|---|---|---|
| `GDPC1` | Real GDP (2017 chained $bn) | Annualised QoQ growth: $400 \times \Delta \ln$ |
| `CPIAUCSL` | CPI All Urban Consumers | Annualised QoQ inflation: $400 \times \Delta \ln$ |
| `FEDFUNDS` | Effective Federal Funds Rate | Level (already in %) |
| `PCECC96` | Real Personal Consumption (2017 $bn) | Log level |
| `DPIC96` | Real Disposable Personal Income (2017 $bn) | Log level |

In [ ]:
# 🎛️ DATA FILE PATH — change this if the file is in a different location
# Google Colab: upload the file first, then set FILE_PATH to the filename.
FILE_PATH = '2026-01-QD.xlsx'

# ── Load raw file ─────────────────────────────────────────────────────────────
# header=None because row 0 is variable names (not a standard pandas header)
df       = pd.read_excel(FILE_PATH, sheet_name='in', header=None)
colnames = df.iloc[0].values    # row 0 = variable names

# ── Locate columns by name ────────────────────────────────────────────────────
# Using np.where makes the code robust to column reordering across FRED-QD vintages
gdp_idx    = np.where(colnames == 'GDPC1')[0][0]
cpi_idx    = np.where(colnames == 'CPIAUCSL')[0][0]
ff_idx     = np.where(colnames == 'FEDFUNDS')[0][0]
cons_idx   = np.where(colnames == 'PCECC96')[0][0]
income_idx = np.where(colnames == 'DPIC96')[0][0]

# iloc[5:] starts at 1959:Q3 — the earliest row we need for growth-rate computation
# pd.to_numeric with errors='coerce' turns any non-numeric entries (e.g. 'NaN' strings) into NaN
gdp    = pd.to_numeric(df.iloc[5:, gdp_idx].values,    errors='coerce')
cpi    = pd.to_numeric(df.iloc[5:, cpi_idx].values,    errors='coerce')
ff     = pd.to_numeric(df.iloc[5:, ff_idx].values,     errors='coerce')
cons   = pd.to_numeric(df.iloc[5:, cons_idx].values,   errors='coerce')
income = pd.to_numeric(df.iloc[5:, income_idx].values, errors='coerce')

# Date index aligned to the data rows (1960-01-01 is the first date after the 1959:Q3 start)
dates = pd.date_range(start='1960-01-01', periods=len(gdp), freq='QE')

print(f'Raw series loaded: {len(gdp)} rows, dates {dates[0].date()} → {dates[-1].date()}')

In [ ]:
# ── Transformations ───────────────────────────────────────────────────────────
# Build a temporary DataFrame to use pandas shift() for the lag operation
temp_df = pd.DataFrame({'GDP': gdp, 'CPI': cpi, 'FF': ff}, index=dates)

# Annualised quarter-on-quarter growth rate: 400 × Δln(X_t)
# Multiplying by 4 converts quarterly growth to an annual rate (standard for US macro)
gdp_growth = 100 * 4 * (np.log(temp_df['GDP']) - np.log(temp_df['GDP'].shift(1)))
inflation  = 100 * 4 * (np.log(temp_df['CPI']) - np.log(temp_df['CPI'].shift(1)))
# Fed funds rate is already in % — no transformation needed

# ── Build analysis DataFrames ─────────────────────────────────────────────────

# VAR dataset: three stationary series for Examples 1 and 3
# dropna() removes the first row (NaN from the log-difference shift)
macro_data = pd.DataFrame(
    {'GDP_Growth': gdp_growth, 'Inflation': inflation, 'FedFunds': ff},
    index=dates).dropna()

# Cointegration dataset: log-levels (I(1)) for Example 2
# We do NOT difference here — Johansen and ECM work in levels
coint_data = pd.DataFrame(
    {'LogConsumption': np.log(cons), 'LogIncome': np.log(income)},
    index=dates).dropna()

var_data = macro_data[['GDP_Growth', 'Inflation', 'FedFunds']].dropna()

# Convenience labels for figure titles
first_q = f"{var_data.index[0].year}:Q{var_data.index[0].quarter}"
last_q  = f"{var_data.index[-1].year}:Q{var_data.index[-1].quarter}"

print(f'VAR dataset : {var_data.shape[0]} obs,  {first_q} – {last_q}')
print(f'Coint dataset: {coint_data.shape[0]} obs')
print()
print(var_data.describe().round(3))

**What you should see:** 264 quarterly observations for the VAR dataset (1960:Q2 – 2026:Q1). GDP growth has a mean around 3% and a large standard deviation (reflecting recessions); inflation averages ~3.6%; the Fed funds rate averages ~4.8% with a wide range reflecting tightening and easing cycles.

In [ ]:
# ── Figure 0: Data overview ───────────────────────────────────────────────────
# Always plot the raw data before estimating any model — visual inspection catches
# outliers, structural breaks, and obvious non-stationarity before formal tests.
fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=True)

plot_specs = [
    ('GDP_Growth', 'Real GDP Growth (annualised, %)', 'steelblue'),
    ('Inflation',  'CPI Inflation (annualised, %)',    'darkred'),
    ('FedFunds',   'Federal Funds Rate (%)',           'darkgreen'),
]
for ax, (col, title, color) in zip(axes, plot_specs):
    ax.plot(var_data.index, var_data[col], color=color, linewidth=0.9)
    ax.axhline(var_data[col].mean(), color='black', linestyle='--',
               linewidth=1.0, alpha=0.6,
               label=f'Mean = {var_data[col].mean():.2f}%')
    ax.axhline(0, color='red', linewidth=0.6, alpha=0.5)
    ax.set_title(title)
    ax.set_ylabel('%')
    ax.legend(loc='upper right')

# Add recession shading (NBER recessions, approximate)
recessions = [
    ('1960-04-01','1961-01-01'), ('1969-10-01','1970-10-01'),
    ('1973-10-01','1975-01-01'), ('1980-01-01','1980-07-01'),
    ('1981-07-01','1982-10-01'), ('1990-07-01','1991-01-01'),
    ('2001-01-01','2001-10-01'), ('2007-10-01','2009-04-01'),
    ('2020-01-01','2020-07-01'),
]
for ax in axes:
    for start, end in recessions:
        ax.axvspan(pd.Timestamp(start), pd.Timestamp(end),
                   alpha=0.12, color='grey')   # grey shading for NBER recessions

axes[-1].set_xlabel('Quarter')
fig.suptitle(f'Figure 0: US Macroeconomic Data — FRED-QD\n'
             f'{first_q} – {last_q}  |  Grey shading = NBER recessions',
             fontsize=13, fontweight='bold')
fig.tight_layout()
path = os.path.join(OUT_DIR, 'fig0_data_overview.png')
fig.savefig(path, dpi=150, bbox_inches='tight')
fig.savefig(path.replace('.png','.pdf'), bbox_inches='tight')
plt.show()
print(f'Saved: {path}')

---
# Example 1 — VAR Estimation
### Lag Selection · Coefficient Estimates · Residual Diagnostics · Granger Causality
*Textbook reference: Sections 2.1–2.4*

## Background

The **Vector Autoregression** (VAR) model of order $p$ writes:

$$Y_t = c + A_1 Y_{t-1} + A_2 Y_{t-2} + \cdots + A_p Y_{t-p} + u_t, \quad u_t \sim WN(0, \Sigma)$$

where $Y_t = (\text{GDP growth}_t, \pi_t, i_t)'$ is a $3\times 1$ vector and each $A_j$ is a $3\times 3$ coefficient matrix. With $p=1$ and 3 variables, we estimate $3 \times (3\times 1 + 1) = 12$ parameters (3 equations × (3 autoregressive coefficients + 1 constant)).

**Why estimate by equation-by-equation OLS?** When all equations have the same regressors (the standard VAR case), OLS on each equation separately is equivalent to GLS on the full system. It is also consistent and asymptotically efficient under Gaussian errors.

### The lag selection problem

The lag order $p$ is chosen by minimising an information criterion:

$$\text{AIC}(p) = \ln|\hat{\Sigma}(p)| + \frac{2}{T} \cdot k^2 p, \qquad
\text{BIC}(p) = \ln|\hat{\Sigma}(p)| + \frac{\ln T}{T} \cdot k^2 p$$

where $k=3$ is the number of variables and $k^2 p$ counts the AR coefficients. BIC's stronger penalty favours more parsimonious models, which is especially important for quarterly data where sample sizes are modest.

**Important note on BIC near-tie:** For this dataset, BIC(1) and BIC(2) differ by less than 0.001. Floating-point differences between MATLAB and Python in the determinant/log computation can cause one platform to prefer $p=1$ and the other $p=2$. We fix $p=1$ for consistency with the textbook, but this is a substantive rather than statistical choice — try re-running with `OPTIMAL_LAG = 2` to check robustness.

In [ ]:
# 🎛️ EXAMPLE 1 PARAMETERS
MAX_LAGS    = 12    # maximum lag order to consider in IC comparison
OPTIMAL_LAG = 1     # textbook specification (BIC near-tied with p=2 — see note above)
                    # try OPTIMAL_LAG = 2 for robustness check

print(f'Max lags for IC comparison: {MAX_LAGS}')
print(f'Estimation lag order fixed at: p = {OPTIMAL_LAG}')

## Step 1 — Lag selection

In [ ]:
# ── Lag selection: compute AIC, BIC, HQIC for p = 1 … MAX_LAGS ───────────────
# We fit VAR(p) for each p, extract the IC, then plot them.
# statsmodels' select_order() also returns the minimising p for each IC.
var_model   = VAR(var_data)              # instantiate VAR model (no estimation yet)
lag_sel     = var_model.select_order(maxlags=MAX_LAGS)   # compute ICs for all lags

print(f'IC-selected lag orders:')
print(f'  AIC → p = {lag_sel.aic}   (tends to over-select in small samples)')
print(f'  BIC → p = {lag_sel.bic}   (strongly penalises extra lags)')
print(f'  HQIC→ p = {lag_sel.hqic}  (intermediate penalty)')

# Compute IC values explicitly across lags for the plot
# (select_order doesn't store all values, so we refit)
lags_range = range(1, MAX_LAGS + 1)
aic_vals   = [var_model.fit(p).aic  for p in lags_range]
bic_vals   = [var_model.fit(p).bic  for p in lags_range]
hqic_vals  = [var_model.fit(p).hqic for p in lags_range]

print(f'\nIC values at p = {OPTIMAL_LAG}:')
print(f'  AIC  = {aic_vals[OPTIMAL_LAG-1]:.4f}')
print(f'  BIC  = {bic_vals[OPTIMAL_LAG-1]:.4f}')
print(f'  HQIC = {hqic_vals[OPTIMAL_LAG-1]:.4f}')

In [ ]:
# ── Figure 1: Lag selection plot ──────────────────────────────────────────────
# Plotting IC values across lags makes the trade-off between fit and parsimony visible.
# Vertical dashed lines mark the IC-minimising lags.
fig, ax = plt.subplots(figsize=(11, 6))

ic_specs = [
    (aic_vals,  lag_sel.aic,  'AIC',  '#27AE60', 'o'),
    (bic_vals,  lag_sel.bic,  'BIC',  '#E74C3C', 's'),
    (hqic_vals, lag_sel.hqic, 'HQIC', '#2980B9', '^'),
]

for vals, opt_lag, label, color, marker in ic_specs:
    ax.plot(list(lags_range), vals, color=color, linewidth=2.0,
            marker=marker, markersize=6, label=label)
    # Vertical line at the IC-minimising lag
    ax.axvline(opt_lag, color=color, linewidth=1.4, linestyle='--', alpha=0.7)

# Mark our chosen lag with a bold vertical line
ax.axvline(OPTIMAL_LAG, color='black', linewidth=2.2, linestyle='-',
           label=f'Textbook choice: p = {OPTIMAL_LAG}')

ax.set_xlabel('Number of Lags ($p$)')
ax.set_ylabel('Information Criterion Value')
ax.set_xticks(list(lags_range))
ax.set_title(f'Figure 1: VAR Lag Order Selection\n'
             f'Real US data: {first_q} – {last_q}\n'
             f'Note: BIC(1) and BIC(2) differ by < 0.001 — near tie',
             fontsize=12)
ax.legend()
fig.tight_layout()
path = os.path.join(OUT_DIR, 'fig1_lag_selection.png')
fig.savefig(path, dpi=150, bbox_inches='tight')
fig.savefig(path.replace('.png','.pdf'), bbox_inches='tight')
plt.show()
print(f'Saved: {path}')

**What to look for:** AIC typically selects more lags than BIC because its penalty term is smaller. The BIC curve should be relatively flat near its minimum — the near-tie between $p=1$ and $p=2$ is visible as the BIC curve barely changes. This flatness is common with quarterly data; when it occurs, additional economic judgment (theory, parsimony) is needed to break the tie.

## Step 2 — VAR estimation

In [ ]:
# ── Fit the VAR(p) model ──────────────────────────────────────────────────────
# OLS estimation equation by equation — equivalent to GLS for a standard VAR
var_result = var_model.fit(OPTIMAL_LAG)
residuals  = var_result.resid    # T × k matrix of reduced-form residuals u_t

# ── Display coefficient table ─────────────────────────────────────────────────
# The summary shows each equation's OLS coefficients with t-stats and p-values
print(var_result.summary())

**Reading the output:** Each block is one equation. `L1.GDP_Growth` means GDP growth lagged one period. Look for which own-lags are significant (AR dynamics) and which cross-variable lags are significant (Granger causality). The Fed funds rate equation in particular captures monetary policy dynamics.

## Step 3 — Residual diagnostics

A correctly specified VAR should have **white noise residuals** in each equation — no remaining autocorrelation, no outliers, and approximately normal distribution. We check all three:

- **Time series plot**: residuals should scatter randomly around zero.
- **Histogram**: should be bell-shaped (no severe skew or heavy tails).
- **ACF of residuals**: all spikes inside the 95% band — no residual autocorrelation.

After the visual check, we apply the **Portmanteau (Ljung-Box) test** for multivariate autocorrelation.

In [ ]:
# ── Figure 2: Residual diagnostics (3 × 3 panel) ─────────────────────────────
# Each row = one VAR equation; columns = time series, histogram, ACF
VAR_NAMES = ['GDP Growth', 'Inflation', 'Fed Funds Rate']

fig, axes = plt.subplots(3, 3, figsize=(14, 11))
fig.suptitle(f'Figure 2: VAR({OPTIMAL_LAG}) Residual Diagnostics\n'
             f'GDP Growth · Inflation · Federal Funds Rate',
             fontsize=13, fontweight='bold')

for i, (col, name) in enumerate(zip(var_data.columns, VAR_NAMES)):
    r = residuals.iloc[:, i]

    # Column 0: Residual time series
    axes[i,0].plot(r.index, r.values, 'steelblue', linewidth=0.7)
    axes[i,0].axhline(0, color='red', linewidth=1.0, linestyle='--')
    axes[i,0].set_title(f'{name}: Residuals over time')
    axes[i,0].set_ylabel('Residual')

    # Column 1: Histogram with fitted normal
    axes[i,1].hist(r.values, bins=28, density=True,
                   alpha=0.7, color='steelblue', edgecolor='white')
    xg = np.linspace(r.mean() - 4*r.std(), r.mean() + 4*r.std(), 200)
    axes[i,1].plot(xg, stats.norm.pdf(xg, r.mean(), r.std()),
                   'r-', linewidth=2.0, label='Normal fit')
    axes[i,1].set_title(f'{name}: Histogram')
    axes[i,1].legend()

    # Column 2: ACF of residuals (should show no significant spikes)
    plot_acf(r, lags=20, ax=axes[i,2], zero=False, alpha=0.05,
             color='steelblue', vlines_kwargs={'colors': 'steelblue'})
    axes[i,2].set_title(f'{name}: ACF of residuals')
    axes[i,2].set_xlabel('Lag')

fig.tight_layout()
path = os.path.join(OUT_DIR, 'fig2_var_diagnostics.png')
fig.savefig(path, dpi=150, bbox_inches='tight')
fig.savefig(path.replace('.png','.pdf'), bbox_inches='tight')
plt.show()
print(f'Saved: {path}')

In [ ]:
# ── Formal diagnostic tests on each equation's residuals ─────────────────────

print(f'Residual diagnostics — VAR({OPTIMAL_LAG}) on real US data\n')
print(f'{"Equation":<16} {"Mean":>8} {"Std":>8} {"Skew":>8} {"Kurt":>8} '
      f'{"JB p":>8} {"DW":>8}')
print('-' * 72)

for i, name in enumerate(VAR_NAMES):
    r = residuals.iloc[:, i]
    jb_stat, jb_p = stats.jarque_bera(r)     # normality test
    dw = durbin_watson(r)                     # first-order autocorrelation test
    print(f'{name:<16} {r.mean():>8.4f} {r.std():>8.4f} '
          f'{stats.skew(r):>8.3f} {stats.kurtosis(r):>8.3f} '
          f'{jb_p:>8.4f} {dw:>8.3f}')

print('\nNote: DW ≈ 2 = no first-order autocorrelation. JB p < 0.05 = non-normal.')

# Portmanteau test — multivariate version (tests all equations jointly)
port_result = var_result.test_whiteness(nlags=10)
print(f'\nPortmanteau (whiteness) test on all residuals jointly:')
print(f'  Statistic = {port_result.test_statistic:.3f}')
print(f'  p-value   = {port_result.pvalue:.4f}')
print(f'  → {"No residual autocorrelation ✓" if port_result.pvalue > 0.05 else "Residual autocorrelation detected ✗"}')

**What to look for:** DW close to 2 in each equation means no first-order autocorrelation. The GDP equation residuals may show excess kurtosis (heavy tails) due to large recessions — this is common with macro data and affects inference but not consistency of OLS. The Portmanteau $p$-value should exceed 0.05 for the model to pass the whiteness test.

## Step 4 — Granger causality

**Granger causality** (Granger, 1969) answers: does knowing the history of $X$ help predict $Y$, *over and above* the information already in the history of $Y$ itself? It is a predictability test, **not** a causal statement in the structural/counterfactual sense.

Formally, $X$ does not Granger-cause $Y$ if:
$$E[Y_{t+1} | \mathcal{F}_t] = E[Y_{t+1} | \mathcal{F}_t \setminus \{X_s, s \leq t\}]$$

In a VAR, this reduces to an $F$-test on whether the coefficients on $X_{t-1}, \ldots, X_{t-p}$ in the $Y$ equation are jointly zero.

In [ ]:
# ── Granger causality tests — all ordered pairs ───────────────────────────────
# grangercausalitytests(data[[Y, X]], maxlag=p) tests whether X Granger-causes Y.
# The convention: the FIRST column is the dependent variable (the one being predicted),
# the SECOND column is the potential Granger-cause.

print(f'Granger Causality Tests — VAR({OPTIMAL_LAG}), real US data')
print(f'H₀: column X does NOT Granger-cause row Y')
print(f'(F-test on whether X lags are jointly zero in the Y equation)\n')
print(f'{"X (cause)":<14} → {"Y (effect)":<14}  {"F-stat":>8}  {"p-value":>8}  {"Sig":>5}')
print('-' * 60)

for y_var in var_data.columns:    # loop over dependent (predicted) variable
    for x_var in var_data.columns:    # loop over potential Granger-cause
        if x_var == y_var:
            continue
        # data ordering: [Y, X] — first col = dependent, second = cause
        gc = grangercausalitytests(
            var_data[[y_var, x_var]],
            maxlag=OPTIMAL_LAG, verbose=False)
        f_stat, p_val, _, _ = gc[OPTIMAL_LAG][0]['ssr_ftest']
        stars = ('***' if p_val < 0.01
                 else ('**' if p_val < 0.05
                       else ('*' if p_val < 0.10 else '')))
        print(f'{x_var:<14} → {y_var:<14}  {float(f_stat):>8.2f}  {float(p_val):>8.4f}  {stars:>5}')

print('-' * 60)
print('Significance: *** p<0.01  ** p<0.05  * p<0.10')
print('\nReminder: "Granger-causes" = "has predictive power for" — NOT structural causality.')

**What to look for:** We expect the Fed funds rate to have predictive content for GDP growth and inflation (monetary policy transmission), and we expect inflation to help predict the Fed funds rate (Taylor rule). Significant Granger causality in both directions between variables is consistent with feedback in the macro system. Note carefully: this tells us about *predictability*, not about structural responses to policy shocks — that requires identification (Example 3).

---
# Example 2 — Cointegration and VECM
### Permanent Income Hypothesis: Consumption and Disposable Income
*Textbook reference: Section 2.5*

## Background

Both log consumption and log income are I(1) — they have unit roots. The Permanent Income Hypothesis (PIH) predicts they share a common stochastic trend:

$$\ln C_t = \alpha + \beta \ln Y_t + z_t, \quad z_t \sim I(0)$$

If this holds, the series are **cointegrated** and we can estimate the Vector Error Correction Model (VECM), which separates short-run dynamics from the long-run equilibrium.

### Johansen's test

Unlike the Engle-Granger two-step (which tests cointegration by residuals from a single OLS equation), Johansen's (1988) method is a full-system MLE approach. It tests sequentially:
- $H_0: r = 0$ (no cointegration) vs $H_1: r \geq 1$
- $H_0: r \leq 1$ vs $H_1: r = 2$

using the **trace statistic** and **maximum eigenvalue statistic**.

**Deterministic specification (det_order):** We use `det_order=1` (Case 4 in Johansen's taxonomy: linear trend in the data, constant in the cointegrating equation). This is appropriate because log consumption and log income both have clear upward trends due to productivity and population growth. Using `det_order=0` would spuriously find $r=2$ because the test would confuse trend-stationarity with a second cointegrating vector.

In [ ]:
# 🎛️ EXAMPLE 2 PARAMETERS
DET_ORDER  = 1    # Johansen deterministic spec: 1 = linear trend in data (Case 4)
                  # Try det_order=0 to see how the wrong spec finds r=2 spuriously
K_AR_DIFF  = 1    # number of lagged differences in the VECM (equivalent to p-1 in VAR(p))

vecm_data = coint_data[['LogConsumption', 'LogIncome']].dropna()
print(f'Cointegration sample: {len(vecm_data)} obs')
print(f'Johansen deterministic spec: det_order={DET_ORDER} (linear trend in data)')

In [ ]:
# ── Step 1: ADF unit root tests on log levels ─────────────────────────────────
# Cointegration requires both series to be I(1).
# We confirm: (a) non-stationary in levels, (b) stationary in first differences.

print('Unit root tests — LEVELS (expect non-stationary: FAIL to reject H₀):')
for col in vecm_data.columns:
    adf = adfuller(vecm_data[col], autolag='AIC')
    print(f'  {col:<18}: ADF stat = {adf[0]:.3f},  p = {adf[1]:.3f}  '
          f'→ {"I(0)" if adf[1]<0.05 else "I(1) — unit root"}')

print('\nUnit root tests — FIRST DIFFERENCES (expect stationary: REJECT H₀):')
for col in vecm_data.columns:
    adf_d = adfuller(vecm_data[col].diff().dropna(), autolag='AIC')
    print(f'  Δ{col:<17}: ADF stat = {adf_d[0]:.3f},  p = {adf_d[1]:.4f}  '
          f'→ {"I(0) ✓" if adf_d[1]<0.05 else "Still non-stationary"}')

In [ ]:
# ── Step 2: Johansen cointegration test ──────────────────────────────────────
# coint_johansen(data, det_order, k_ar_diff) runs the Johansen trace and max-eigenvalue tests.
# det_order=1 includes a linear trend in the data and a constant in the CE.
# k_ar_diff=1 means the VECM has 1 lagged difference (equivalent to a VAR(2) in levels).
johansen = coint_johansen(vecm_data, det_order=DET_ORDER, k_ar_diff=K_AR_DIFF)

print('Johansen TRACE Test (H₀: rank ≤ r  vs  H₁: rank > r):')
print(f'  {"H₀":>6}  {"Statistic":>12}  {"90% CV":>8}  {"95% CV":>8}  {"99% CV":>8}  {"Conclusion":>30}')
print('-' * 80)
for i in range(2):
    stat = johansen.lr1[i]
    cv90, cv95, cv99 = johansen.cvt[i]
    concl = 'Reject H₀ at 5%' if stat > cv95 else 'Fail to reject'
    print(f'  r ≤ {i:>1}  {stat:>12.2f}  {cv90:>8.2f}  {cv95:>8.2f}  {cv99:>8.2f}  {concl:>30}')

print('\nJohansen MAX-EIGENVALUE Test (H₀: rank = r  vs  H₁: rank = r+1):')
print(f'  {"H₀":>6}  {"Statistic":>12}  {"90% CV":>8}  {"95% CV":>8}  {"99% CV":>8}  {"Conclusion":>30}')
print('-' * 80)
for i in range(2):
    stat = johansen.lr2[i]
    cv90, cv95, cv99 = johansen.cvm[i]
    concl = 'Reject H₀ at 5%' if stat > cv95 else 'Fail to reject'
    print(f'  r = {i:>1}   {stat:>12.2f}  {cv90:>8.2f}  {cv95:>8.2f}  {cv99:>8.2f}  {concl:>30}')

print('\nConclusion: both tests find r = 1 cointegrating vector → one common stochastic trend.')

In [ ]:
# ── Step 3: OLS long-run relationship ────────────────────────────────────────
# We estimate the cointegrating vector by OLS (Engle-Granger first step).
# OLS on I(1) variables is valid when they are cointegrated (super-consistency):
# the estimate converges at rate T rather than √T.
# We use OLS rather than reading the Johansen eigenvector directly to avoid
# the sign/normalisation ambiguity in the VECM beta matrix.
X_lr  = np.column_stack([np.ones(len(vecm_data)),
                          vecm_data['LogIncome'].values])
b_lr  = np.linalg.lstsq(X_lr, vecm_data['LogConsumption'].values, rcond=None)[0]
ols_intercept, ols_slope = b_lr

# Cointegrating residual = deviation from long-run equilibrium
coint_resid    = vecm_data['LogConsumption'].values - (ols_intercept + ols_slope * vecm_data['LogIncome'].values)
coint_resid_std = (coint_resid - coint_resid.mean()) / coint_resid.std()   # standardise for plot

print(f'Long-run OLS: ln C = {ols_intercept:.4f} + {ols_slope:.4f} · ln Y')
print(f'  Intercept : {ols_intercept:.4f}  (≈ log of base consumption level)')
print(f'  Long-run MPC: {ols_slope:.4f}  (1% rise in income → {ols_slope:.2f}% rise in consumption)')
print(f'\nNote: OLS slope close to 1.0 is consistent with the PIH (unit elasticity).')

# ADF on the cointegrating residual — use Engle-Granger critical values
adf_cr = adfuller(coint_resid, autolag='AIC', regression='n')
eg_cv5 = -3.34    # Engle-Granger 5% CV for 2 variables
print(f'\nADF on cointegrating residual (Engle-Granger step 2):')
print(f'  ADF stat = {adf_cr[0]:.4f},  EG 5% CV = {eg_cv5:.2f}')
print(f'  Residual is {"stationary ✓ — cointegration confirmed" if adf_cr[0] < eg_cv5 else "non-stationary ✗"}')

In [ ]:
# ── Figure 3: Cointegration analysis ─────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(13, 11))
fig.suptitle(f'Figure 3: Cointegration Analysis — Log Consumption and Log Income\n'
             f'Real US data: {first_q} – {last_q}',
             fontsize=13, fontweight='bold')

# Panel (a): Both I(1) series in levels — should move closely together
axes[0].plot(vecm_data.index, vecm_data['LogConsumption'],
             color='steelblue', linewidth=1.2, label='Log Consumption')
axes[0].plot(vecm_data.index, vecm_data['LogIncome'],
             color='darkorange', linewidth=1.2, label='Log Income')
axes[0].set_title('(a) Log Consumption and Log Income (I(1) levels — common stochastic trend)')
axes[0].set_ylabel('Log Level')
axes[0].legend(loc='upper left')

# Panel (b): Scatter with cointegrating regression line
axes[1].scatter(vecm_data['LogIncome'], vecm_data['LogConsumption'],
                alpha=0.35, s=14, color='purple', edgecolors='none')
y_rng = np.linspace(vecm_data['LogIncome'].min(), vecm_data['LogIncome'].max(), 200)
axes[1].plot(y_rng, ols_intercept + ols_slope * y_rng,
             color='red', linewidth=2.2,
             label=f'$\\ln C = {ols_intercept:.3f} + {ols_slope:.3f} \\cdot \\ln Y$')
axes[1].set_xlabel('Log Income')
axes[1].set_ylabel('Log Consumption')
axes[1].set_title(f'(b) Long-run relationship: slope = {ols_slope:.3f} (consistent with PIH)')
axes[1].legend(loc='upper left')

# Panel (c): Standardised equilibrium error — should be stationary (mean-reverting)
axes[2].plot(vecm_data.index, coint_resid_std,
             color='darkgreen', linewidth=0.9)
axes[2].axhline(0, color='red', linestyle='--', linewidth=1.5)
axes[2].fill_between(vecm_data.index, 0, coint_resid_std, alpha=0.2, color='darkgreen')
# Add ±1.5σ reference lines
for lev in [1.5, -1.5]:
    axes[2].axhline(lev, color='orange', linestyle=':', linewidth=1.1)
axes[2].set_title('(c) Equilibrium error $z_t = \\ln C_t - \\hat{\\alpha} - \\hat{\\beta}\\ln Y_t$ (standardised)\n'
                  '(mean-reverting = cointegration confirmed; sustained deviations = consumption-saving episodes)')
axes[2].set_ylabel('Std deviations from equilibrium')
axes[2].set_xlabel('Quarter')

fig.tight_layout()
path = os.path.join(OUT_DIR, 'fig3_cointegration.png')
fig.savefig(path, dpi=150, bbox_inches='tight')
fig.savefig(path.replace('.png','.pdf'), bbox_inches='tight')
plt.show()
print(f'Saved: {path}')

**What to look for in panel (a):** Both series trend upward together — the visual signature of cointegration. They never permanently diverge. In panel (b), the scatter is tight around the regression line, reflecting the long-run equilibrium. In panel (c), the standardised equilibrium error oscillates around zero — it is stationary. Large deviations (e.g. during 2008–2009 or 2020) correspond to periods of elevated saving relative to income.

---
# Example 3 — The Identification Problem
### Why Reduced-Form IRFs Cannot Be Given Structural Interpretation
*Textbook reference: Section 2.6*

## Background

The VAR estimated in Example 1 gives us the **reduced-form** innovations $u_t$, related to the structural shocks $\varepsilon_t$ by:

$$u_t = B_0^{-1} \varepsilon_t, \quad \varepsilon_t \sim WN(0, I_k)$$

The matrix $B_0^{-1}$ is **unknown** — it translates structural shocks (e.g. a monetary policy shock) into the correlated reduced-form innovations we observe. Without knowledge of $B_0$, we cannot answer questions like: *what happens to GDP when the Fed raises rates by 1 percentage point?*

**The identification problem in two steps:**
1. The residuals $u_t$ are correlated across equations (shown in the correlation heatmap below).
2. A shock to any one reduced-form innovation therefore contains contributions from *multiple* structural shocks. The reduced-form IRFs (also shown below) conflate these — they are not structural responses.

**What comes next in the textbook (Chapter 3):** Structural VARs (SVARs) impose restrictions on $B_0$ — Cholesky ordering, sign restrictions, external instruments — to recover the structural shocks from $u_t$.

In [ ]:
# ── Step 1: Show that reduced-form residuals are correlated ───────────────────
# If u_t were orthogonal (diagonal Σ), each innovation would correspond to a single
# structural shock. The off-diagonal elements of Σ are the root of the problem.

corr_mat  = residuals.corr()    # k × k correlation matrix of reduced-form innovations
sigma_hat = residuals.cov()     # k × k covariance matrix Σ̂

print('Reduced-form covariance matrix Σ̂ (full matrix — off-diagonals reflect identification problem):')
print(sigma_hat.round(4))
print()
print('Correlation matrix of reduced-form innovations:')
print(corr_mat.round(4))
print()
print('Key off-diagonal correlations:')
for i, n1 in enumerate(VAR_NAMES):
    for j, n2 in enumerate(VAR_NAMES):
        if j > i:
            c = corr_mat.iloc[i, j]
            print(f'  {n1} ↔ {n2}: {c:.4f}   '
                  f'{"→ substantial correlation — shocks are NOT separable" if abs(c) > 0.2 else "→ modest correlation"}')

In [ ]:
# ── Figure 4: Innovation correlation heatmap ──────────────────────────────────
# A visual display of Σ̂ / (σ_i × σ_j) — each cell is the correlation between
# the reduced-form innovations in two equations.
fig, ax = plt.subplots(figsize=(8, 6))

corr_vals = corr_mat.values
# RdBu_r: red = positive, blue = negative, white = zero
im = ax.imshow(corr_vals, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')

ax.set_xticks(range(3))
ax.set_yticks(range(3))
ax.set_xticklabels(VAR_NAMES, fontsize=11)
ax.set_yticklabels(VAR_NAMES, fontsize=11)

for i in range(3):
    for j in range(3):
        clr = 'white' if abs(corr_vals[i, j]) > 0.5 else 'black'
        ax.text(j, i, f'{corr_vals[i,j]:.3f}',
                ha='center', va='center',
                color=clr, fontsize=14, fontweight='bold')

plt.colorbar(im, ax=ax, label='Correlation coefficient')
ax.set_title('Figure 4: Correlation of Reduced-Form Innovations $u_t$\n'
             'Off-diagonal ≠ 0 → shocks are mixed → identification required',
             fontsize=12, fontweight='bold')
fig.tight_layout()
path = os.path.join(OUT_DIR, 'fig4_innovation_correlation.png')
fig.savefig(path, dpi=150, bbox_inches='tight')
fig.savefig(path.replace('.png','.pdf'), bbox_inches='tight')
plt.show()
print(f'Saved: {path}')

**What to look for:** Any off-diagonal correlation different from zero means the reduced-form shocks are not structural. A positive correlation between the GDP growth and inflation innovations, for example, means a "GDP shock" is really a mixture of supply, demand, and monetary shocks. The heatmap makes this contamination visible before we attempt identification.

## Reduced-form IRFs — and why they mislead

An **impulse response function** (IRF) traces the dynamic effect of a shock to one variable on all variables over a horizon of $H$ quarters. For a reduced-form VAR, the IRF at horizon $h$ is:

$$\Phi_h = A_1 \Phi_{h-1} + A_2 \Phi_{h-2} + \cdots + A_p \Phi_{h-p}, \quad \Phi_0 = I_k$$

The problem: $\Phi_h$ traces the response to a unit innovation in $u_t$ — but $u_t$ is a mixture of structural shocks. The "response of GDP to a Fed funds innovation" is not the response to a monetary policy shock; it is the response to whatever cocktail of structural shocks happens to move the Fed funds rate in the data.

This is why the figure below carries an explicit ⚠ warning.

In [ ]:
# ── Compute reduced-form IRFs ─────────────────────────────────────────────────
# irf(periods=H) computes Φ_0, Φ_1, ..., Φ_H
# These are the REDUCED-FORM moving-average coefficients, NOT structural responses
IRF_HORIZON = 20    # 20 quarters = 5 years ahead

irf_obj = var_result.irf(periods=IRF_HORIZON)
# irf_obj.irfs has shape (H+1, k, k)
# irf_obj.irfs[h, j, i] = response of variable j at horizon h to a unit innovation in i

print(f'Reduced-form IRF computed for horizon h = 0 to {IRF_HORIZON} quarters.')
print(f'Shape of irf array: {irf_obj.irfs.shape}  (horizons × responses × impulses)')
print(f'\nReminder: these are responses to REDUCED-FORM innovations, not structural shocks.')

In [ ]:
# ── Figure 5: Reduced-form IRFs with identification warning ───────────────────
var_labels  = list(var_data.columns)
hor_axis    = np.arange(IRF_HORIZON + 1)

fig, axes = plt.subplots(3, 3, figsize=(14, 12))
fig.suptitle(
    'Figure 5: Reduced-Form Impulse Response Functions\n'
    '⚠  NOT economically identified — statistical responses only',
    fontsize=13, fontweight='bold', color='darkred'
)

for i, impulse_var in enumerate(var_labels):   # impulse (column)
    for j, response_var in enumerate(var_labels):   # response (row)
        ax  = axes[j, i]
        irf = irf_obj.irfs[:, j, i]    # horizon × 1 array

        ax.plot(hor_axis, irf, 'steelblue', linewidth=2.0)
        ax.axhline(0, color='black', linewidth=0.8)
        ax.fill_between(hor_axis, 0, irf, alpha=0.18, color='steelblue')

        ax.set_title(f'Response of {response_var}\nto {impulse_var} innovation',
                     fontsize=9)
        if j == 2:
            ax.set_xlabel('Quarters')
        ax.set_ylabel('Response', fontsize=9)

        # Add a small warning badge on the diagonal (own-shock) panels
        if i == j:
            ax.set_facecolor('#FFF8E1')   # faint yellow background on diagonal

# Bottom footnote explaining why these IRFs should not be interpreted structurally
fig.text(
    0.5, 0.005,
    'These are REDUCED-FORM IRFs to correlated innovations.  '
    'They conflate multiple structural shocks.  '
    'Identification (Cholesky, sign restrictions, external instruments) is required before '
    'attaching causal interpretation.  See Chapter 3.',
    ha='center', fontsize=10, style='italic', color='darkred',
    bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9, edgecolor='orange')
)
fig.tight_layout(rect=[0, 0.04, 1, 0.96])
path = os.path.join(OUT_DIR, 'fig5_reduced_form_irfs.png')
fig.savefig(path, dpi=150, bbox_inches='tight')
fig.savefig(path.replace('.png','.pdf'), bbox_inches='tight')
plt.show()
print(f'Saved: {path}')

**What to look for — and what NOT to conclude:** The 3×3 grid shows how each variable responds over 20 quarters to a unit innovation in each other variable. Note in particular the response of GDP growth to a Fed funds innovation — it may show a *positive* or puzzling response. **This is the price puzzle**: under Cholesky identification with the Fed funds rate ordered last, the reduced-form "monetary policy shock" contains an endogenous policy response to future inflation that has not been filtered out. Chapter 3 shows how proper identification resolves this.

---
## Summary — All three examples

| Example | Key finding | Critical diagnostic |
|---|---|---|
| **1 — VAR** | BIC selects $p=1$ (near-tied with $p=2$); Fed funds rate Granger-causes GDP and inflation | Portmanteau test passes; DW ≈ 2 in all equations |
| **2 — Cointegration** | Johansen finds $r=1$; long-run MPC ≈ 0.99 (consistent with PIH) | ADF on cointegrating residual < EG 5% CV |
| **3 — Identification** | Off-diagonal innovation correlations ≠ 0; reduced-form IRFs are not causal | Heatmap shows correlation; price puzzle visible in IRFs |

## Exercises

**Exercise 1.1 (VAR lag selection):** Change `MAX_LAGS` to 8 and re-run. Does the IC-selected lag order change? Does the BIC near-tie become more or less obvious?

**Exercise 1.2 (Granger causality):** Re-estimate with `OPTIMAL_LAG = 2`. Do the Granger causality results change materially? Which direction of causality is most sensitive to the lag order?

**Exercise 2.1 (Johansen spec):** Change `DET_ORDER` from 1 to 0 and re-run Example 2. Does the test now spuriously find $r=2$? This illustrates why the deterministic specification matters.

**Exercise 2.2 (Alternative pair):** Replace `LogConsumption`/`LogIncome` with two other I(1) series from the FRED-QD file (e.g. log real GDP and log real consumption). Do you find cointegration? Is the long-run elasticity close to 1?

**Exercise 3.1 (Identification preview):** Apply a Cholesky decomposition to $\hat{\Sigma}$ (lower triangular Cholesky factor `P = np.linalg.cholesky(sigma_hat)`). Compute the orthogonalised IRFs as `P @ irf_obj.irfs`. Does the response of GDP to the Fed funds shock now look more plausible?

**Exercise 3.2 (Price puzzle):** With Cholesky ordering (GDP, Inflation, FedFunds) — the standard CEE ordering — plot the response of inflation to a positive Fed funds shock. Does it go negative (correct sign) or positive (price puzzle)? Try reversing the ordering.

---
*Macroeconometrics* | Alessia Paccagnini  
Companion code — Chapter 2 VAR Examples | Last updated: March 2026  
Data: FRED-QD 2026-01-QD, McCracken & Ng